# Бинарная классификация "таблица - не таблица"

## Подготовка размеченных данных

Данные были размечены инструментом CVAT

In [1]:
import pandas as pd
df = pd.read_xml('annotations.xml')
df

,version,job,dumped,id,name,width,height,tag
0,1.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,\n,2026-04-16 05:11:50.370002+00:00,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,0.0,photo_1000@06-11-2023_12-51-41.jpg,1075.0,1280.0,\n
3,NaN,NaN,NaN,1.0,photo_1001@08-11-2023_11-31-02.jpg,1075.0,1280.0,\n
4,NaN,NaN,NaN,2.0,photo_1002@08-11-2023_11-31-02.jpg,1075.0,1280.0,\n
...,...,...,...,...,...,...,...,...
3449,NaN,NaN,NaN,3447.0,photo_997@03-11-2023_13-43-26.jpg,1280.0,1280.0,\n
3450,NaN,NaN,NaN,3448.0,photo_998@05-11-2023_11-12-34.jpg,1280.0,1065.0,\n
3451,NaN,NaN,NaN,3449.0,photo_999@06-11-2023_12-51-41.jpg,1075.0,1280.0,\n
3452,NaN,NaN,NaN,3450.0,photo_99@25-07-2022_12-52-49.jpg,1280.0,1280.0,\n


Загрузить напрямую из XML не получилось. Попросил DeepSeek написать функцию для парсинга XML файла в словарь

In [2]:
import xml.etree.ElementTree as ET

tree = ET.parse('annotations.xml')
root = tree.getroot()

In [3]:
def etree_to_dict(t):
    d = {t.tag: {} if t.attrib else None}
    children = list(t)
    if children:
        dd = {}
        for dc in map(etree_to_dict, children):
            for k, v in dc.items():
                if k in dd:
                    if not isinstance(dd[k], list):
                        dd[k] = [dd[k]]
                    dd[k].append(v)
                else:
                    dd[k] = v
        d = {t.tag: dd}
    if t.attrib:
        d[t.tag].update(('@' + k, v) for k, v in t.attrib.items())
    if t.text and t.text.strip():
        if children or t.attrib:
            d[t.tag]['#text'] = t.text.strip()
        else:
            d[t.tag] = t.text.strip()
    return d

root = ET.parse('annotations.xml').getroot()
full_dict = etree_to_dict(root)

Выгрузка в JSON чтобы просмотреть получившийся словарь

In [ ]:
import json
with open('labels.json', 'w', encoding='utf8') as jsonfile:
    json.dump(full_dict, jsonfile, indent=4, ensure_ascii=False)

Так как тэг хранится во вложенном словаре, надо его оттуда вытащить. Так же некоторые фотографии остались неразмеченными, это тоже надо учесть

In [ ]:
df = pd.DataFrame(full_dict['annotations']['image'])
df['tag'] = df['tag'].apply(lambda x: x.get('@label') if isinstance(x, dict) else x)
df

,tag,@id,@name,@width,@height
0,table,0,photo_1000@06-11-2023_12-51-41.jpg,1075,1280
1,table,1,photo_1001@08-11-2023_11-31-02.jpg,1075,1280
2,table,2,photo_1002@08-11-2023_11-31-02.jpg,1075,1280
3,table,3,photo_1003@09-11-2023_11-31-47.jpg,1075,1280
4,table,4,photo_1004@09-11-2023_11-31-47.jpg,1075,1280
...,...,...,...,...,...
3447,not_table,3447,photo_997@03-11-2023_13-43-26.jpg,1280,1280
3448,not_table,3448,photo_998@05-11-2023_11-12-34.jpg,1280,1065
3449,table,3449,photo_999@06-11-2023_12-51-41.jpg,1075,1280
3450,not_table,3450,photo_99@25-07-2022_12-52-49.jpg,1280,1280


Выгрузим полученный фрейм в Excel и заполним пропуски вручную

In [5]:
df.to_excel('labels.xlsx')